# Φάση Γ: Classification Models

**Υπεύθυνος:** ML Engineer

**Μοντέλα:**
1. Random Forest
2. SVM


In [7]:
# Φάση Γ: Classification Models (Προσθήκη Μοντέλων για Imbalance)
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.classification import (RandomForestClassifier, LinearSVC, NaiveBayes, 
                                       MultilayerPerceptronClassifier, GBTClassifier, LogisticRegression)

print("Εκκίνηση SparkSession...")
spark = SparkSession.builder.appName("Stroke_Classification").master("local[*]").getOrCreate()

train_gold = spark.read.parquet("../data/train_gold.parquet")
test_gold = spark.read.parquet("../data/test_gold.parquet")

train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))

# --- ADVANCED TECHNIQUE 1: Cost-Sensitive Learning (Βάρη Κλάσεων) ---
stroke_count = train_gold.filter(F.col("stroke") == 1.0).count()
total_count = train_gold.count()
balance_ratio = (total_count - stroke_count) / stroke_count
train_gold = train_gold.withColumn("class_weight", F.when(F.col("stroke") == 1.0, balance_ratio).otherwise(1.0))

# --- ADVANCED TECHNIQUE 2: Random Undersampling (Data-Centric Approach) ---
# Φτιάχνουμε ένα υποσύνολο όπου οι Υγιείς είναι ακριβώς όσοι και τα Εγκεφαλικά (50-50)
minority_df = train_gold.filter(F.col("stroke") == 1.0)
majority_df = train_gold.filter(F.col("stroke") == 0.0).sample(withReplacement=False, fraction=(stroke_count / (total_count - stroke_count)), seed=42)
train_undersampled = minority_df.union(majority_df)

# ==========================================
# 1. ΤΑ 4 ΑΡΧΙΚΑ ΣΟΥ ΜΟΝΤΕΛΑ
# ==========================================
print("Εκπαίδευση 4 Αρχικών Μοντέλων...")
rf = RandomForestClassifier(featuresCol="features", labelCol="stroke", numTrees=100, seed=42)
rf_preds = rf.fit(train_gold).transform(test_gold)

svm = LinearSVC(featuresCol="features", labelCol="stroke", maxIter=100)
svm_preds = svm.fit(train_gold).transform(test_gold)

nb = NaiveBayes(featuresCol="features", labelCol="stroke")
nb_preds = nb.fit(train_gold).transform(test_gold)

input_size = len(train_gold.select("features").first()[0])
mlp = MultilayerPerceptronClassifier(maxIter=100, layers=[input_size, 16, 8, 2], seed=42, featuresCol="features", labelCol="stroke")
mlp_preds = mlp.fit(train_gold).transform(test_gold)


# ==========================================
# 2. ΤΑ ΝΕΑ ΜΟΝΤΕΛΑ (ΕΙΔΙΚΑ ΓΙΑ IMBALANCE)
# ==========================================
print("Εκπαίδευση Νέων Μοντέλων (GBT, Logistic Regression, Undersampled RF)...")

# Νέο Μοντέλο 1: GBT (Gradient Boosted Trees)
gbt = GBTClassifier(featuresCol="features", labelCol="stroke", maxIter=50, maxDepth=4, seed=42)
gbt_preds = gbt.fit(train_gold).transform(test_gold)

# Νέο Μοντέλο 2: Logistic Regression (Με Class Weights)
lr = LogisticRegression(featuresCol="features", labelCol="stroke", weightCol="class_weight", maxIter=100)
lr_preds = lr.fit(train_gold).transform(test_gold)

# Νέο Μοντέλο 3: Random Forest εκπαιδευμένο στο 50-50 Undersampled Dataset
rf_under = RandomForestClassifier(featuresCol="features", labelCol="stroke", numTrees=100, seed=42)
rf_under_preds = rf_under.fit(train_undersampled).transform(test_gold)


# --- ΑΠΟΘΗΚΕΥΣΗ ΟΛΩΝ ---
print("Αποθήκευση προβλέψεων...")
rf_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/rf_predictions.parquet")
svm_preds.select("stroke", "prediction", "rawPrediction").write.mode("overwrite").parquet("../data/svm_predictions.parquet")
nb_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/nb_predictions.parquet")
mlp_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/mlp_predictions.parquet")
# Νέα
gbt_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/gbt_predictions.parquet")
lr_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/lr_predictions.parquet")
rf_under_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/rf_under_predictions.parquet")

spark.stop()

Εκκίνηση SparkSession...
Εκπαίδευση 4 Αρχικών Μοντέλων...
Εκπαίδευση Νέων Μοντέλων (GBT, Logistic Regression, Undersampled RF)...
Αποθήκευση προβλέψεων...
